# Hugging Face Transformers + PyTorch inference

Run Qwen2.5-0.5B locally through Hugging Face Transformers on PyTorch. Same idea as the Ollama comparison: 3 prompts, measured latency + tokens/sec.

Install first (one time): `pip3 install torch transformers accelerate`

In [ ]:
import json, time
import torch
from transformers import pipeline

print(f"torch {torch.__version__}")
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B"
pipe = pipeline("text-generation", model=MODEL_ID, torch_dtype=torch.float32, device=device)
tokenizer = pipe.tokenizer
print("loaded:", MODEL_ID)

In [ ]:
PROMPTS = {
    "factual": "In one sentence, what is the capital of Australia?",
    "reasoning": "A store sells apples at $2 each and oranges at $3 each. If Maya buys 4 apples and 3 oranges and pays with a $20 bill, how much change does she get? Show the steps.",
    "creative": "Write a two-line poem about rain on a city street at night.",
}

results = {}
for name, prompt in PROMPTS.items():
    start = time.time()
    out = pipe(prompt, max_new_tokens=80, do_sample=False, return_full_text=False)
    seconds = time.time() - start
    text = out[0]["generated_text"].strip()
    gen_tokens = len(tokenizer.encode(text))
    tps = gen_tokens / seconds if seconds > 0 else 0
    results[name] = {"seconds": round(seconds, 2), "generated_tokens": gen_tokens, "tokens_per_sec": round(tps, 1), "output": text}
    print(f"[{name}] {seconds:.1f}s | {gen_tokens} tok | {tps:.1f} tok/s")
    print(f"  -> {text[:200]}")

In [ ]:
with open("hf-results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("saved to hf-results.json")
results